In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import os
from pyunicorn.timeseries import RecurrencePlot
import logging
logging.basicConfig(level=logging.CRITICAL)

In [ ]:
FILE_NAME = r"d:\Pruebas%20BCI\SeñalesProcesadas10Hz2\MauricioServin\S3\5deTorquePre.csv"
band = "mu"

selected_channels = ["C3", "CP3", "C1", "Pz", "Oz", "C4", "CP4", "C2", "FCz", "Cz", "P3", "P4", "FC3", "FC4", "O1", "O2"]

data = pd.read_csv(FILE_NAME)
binary_signal = data["Binaria"]
selected_columns = [f"{band}_" + ch for ch in selected_channels]
data = data[selected_columns] / data[selected_columns].max()

data.shape

(2632, 16)

In [ ]:
def get_recurrence_plots(data, binary_signal):
    """
    Generate recurrence plots for the given data and binary signal.
    
    Parameters:
    - data: DataFrame containing the time series data.
    - binary_signal: Series containing the binary signal associated with the data.
    
    Returns:
    - recurrence_plots: List of RecurrencePlot objects.
    - binary_signal_2: List of binary signal values associated with each recurrence plot.
    """
    win_len = 40
    step = int(win_len * (1 - 0.70))

    recurrence_plots = []
    binary_signal_2 = []
    rr = []
    for i in range(0, len(binary_signal) - win_len + 1, step):
        window_data = data.values[i:i + win_len, :]
        binary_signal_2.append(binary_signal[i])  # etiqueta o valor asociado al inicio de la ventana
        rp = RecurrencePlot(
            window_data,
            threshold=data.std().mean() * 1,  # 0.12 * 2 como en tu código original
            #tau=3,
            epsilon='distance',
            metric="euclidean",
            dim=5  # Dimensión del espacio de fase
        )
        recurrence_plots.append(rp)

    
        rr.append(rp.diag_entropy())
    
    df = pd.DataFrame()
    df["binary_signal"] = binary_signal_2
    df["recurrence_rate"] = rr

    return rr, binary_signal_2

In [ ]:
%matplotlib qt

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

rr_dict = {}
bin_dict = {}

plt.figure(figsize=(30, 4 * len(data.columns)))  # Ajusta altura según número de columnas

for col in data.columns:
    rr, binary_signal_2 = get_recurrence_plots(data[[col]], binary_signal)

    # Guardar en diccionarios
    rr_dict[col] = rr
    bin_dict[col] = binary_signal_2

    # Visualización
    plt.subplot(len(data.columns), 1, data.columns.get_loc(col) + 1)
    plt.plot(rr, label=col, linewidth=2, color='royalblue')

    binary_signal_2 = np.array(binary_signal_2)
    plt.fill_between(
        range(len(binary_signal_2)),
        0,
        1,
        where=binary_signal_2 > 0.5,
        color='red',
        alpha=0.2,
        step='pre',
        label="Actividad"
    )

    plt.title(f"{col}", fontsize=16)
    plt.xlabel("Frame")
    plt.ylabel("Recurrence Rate")
    plt.legend(loc="upper right")
    plt.grid(True, alpha=0.3)

plt.tight_layout()

# Convertir a DataFrames
df_rr = pd.DataFrame(rr_dict)
df_bin = pd.DataFrame(bin_dict)


Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...
Calculating the euclidean distance matrix...
Calculating recurrence plot at fixed threshold...


In [ ]:
df_rr["binary_signal"] = df_bin[f"{band}_Cz"]

# Filtrar filas con actividad (binary_signal == 1)
df_rr_active = df_rr[df_rr["binary_signal"] == 1].drop(columns=["binary_signal"])

# Filtrar filas con reposo (binary_signal == 0)
df_rr_rest = df_rr[df_rr["binary_signal"] == 0].drop(columns=["binary_signal"])


In [ ]:
import mne

In [ ]:
ch_types = ["eeg"] * 16
info = mne.create_info(list(selected_channels), sfreq=10, ch_types=ch_types)
info.set_montage('standard_1020')

Measurement date,Unknown
Experimenter,Unknown
Participant,Unknown
Digitized points,19 points
Good channels,16 EEG
Bad channels,None
EOG channels,Not available
ECG channels,Not available
Sampling frequency,10.00 Hz
Highpass,0.00 Hz
Lowpass,5.00 Hz


In [ ]:
df_rr_active.values.mean(axis=0)

array([2.57357832, 2.51625714, 2.46860971, 2.63287953, 2.57535246,
       2.47761043, 2.48279922, 2.44263647, 2.42976804, 2.4594962 ,
       2.53940626, 2.50424578, 2.38359013, 2.501878  , 2.45515748,
       2.53143565])

In [ ]:
# Calcular valores mínimos y máximos conjuntos
combined = np.vstack([df_rr_active.values.mean(axis=0), df_rr_rest.values.mean(axis=0)])
mini = combined.min()
maxi = combined.max()

# Graficar topomaps con el mismo vlim
fig, axes = plt.subplots(1, 2, figsize=(15, 5))

im, _ = mne.viz.plot_topomap(
    df_rr_active.values.mean(axis=0), 
    info, 
    axes=axes[0], 
    cmap="jet", 
    show=False, 
    extrapolate='local', 
    vlim=(mini, maxi)
)
axes[0].set_title("Actividad")

im, _ = mne.viz.plot_topomap(
    df_rr_rest.values.mean(axis=0), 
    info, 
    axes=axes[1], 
    cmap="jet", 
    show=False, 
    extrapolate='local', 
    vlim=(mini, maxi)
)


axes[1].set_title("Reposo")




plt.tight_layout()
plt.show()

In [ ]:
im, _ = mne.viz.plot_topomap(
     df_rr_active.values.mean(axis=0) - df_rr_rest.values.mean(axis=0), 
    info, 
    
    cmap="jet", 
    show=False, 
    extrapolate='local', 
    
)
